# BrickScanner DB_Setup

Purpose: Initialize Delta tables and seed default rules.

In [ ]:
import logging
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, LongType

from brickscanner.config import DB_TABLES, CATALOG_NAME, SCHEMA_NAME

logger = logging.getLogger("[db_setup]")

## Setup Tables

In [ ]:
def setup_tables(spark):
    """
    Create genai_rule_patterns and genai_output tables with proper schema and constraints.
    
    Args:
        spark: SparkSession
    """
    try:
        # Create catalog and schema if needed
        spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
        
        # Create genai_rule_patterns table
        rule_patterns_table = DB_TABLES["genai_rule_patterns"]
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {rule_patterns_table} (
                rule_id BIGINT GENERATED ALWAYS AS IDENTITY,
                rule_name STRING NOT NULL,
                rule_description STRING,
                rule_category STRING,
                rule_sample STRING,
                is_enabled BOOLEAN DEFAULT true,
                PRIMARY KEY (rule_id)
            )
            USING DELTA
            TBLPROPERTIES (
                'delta.minReaderVersion' = '3',
                'delta.minWriterVersion' = '7',
                'delta.enableDeletionVectors' = 'true'
            )
        """)
        logger.info(f"Created table: {rule_patterns_table}")
        
        # Create genai_output table
        output_table = DB_TABLES["genai_output"]
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {output_table} (
                genai_output_id BIGINT GENERATED ALWAYS AS IDENTITY,
                notebook_id STRING,
                notebook_name STRING,
                notebook_path STRING NOT NULL,
                notebook_url STRING,
                rule_name STRING NOT NULL,
                comment STRING,
                severity STRING NOT NULL,
                matching_content STRING,
                batch_id BIGINT,
                created_by STRING DEFAULT current_user(),
                created_ts TIMESTAMP DEFAULT current_timestamp(),
                PRIMARY KEY (genai_output_id)
            )
            USING DELTA
            TBLPROPERTIES (
                'delta.minReaderVersion' = '3',
                'delta.minWriterVersion' = '7',
                'delta.enableDeletionVectors' = 'true'
            )
        """)
        logger.info(f"Created table: {output_table}")
        
    except Exception as e:
        logger.error(f"Failed to create tables: {e}")
        raise

## Insert Default Rules

In [ ]:
def insert_rules(spark):
    """
    Seed default rules into genai_rule_patterns table.
    
    Args:
        spark: SparkSession
    """
    table_name = DB_TABLES["genai_rule_patterns"]
    
    default_rules = [
        ("unintended-collect", "Pyspark-Coding", "Avoid unnecessary collect()", "df.collect()"),
        ("unintended-stmt", "Pyspark-Coding", "Avoid display, show, print, count in production", "display(df)"),
        ("pandas", "Pyspark-Coding", "Avoid pandas conversion for large-scale processing", "df.toPandas()"),
        ("pandas-udf", "Pyspark-Coding", "Avoid Python UDFs when built-in Spark functions suffice", "@pandas_udf"),
        ("checking-empty", "Pyspark-Coding", "Avoid df.count()==0; prefer isEmpty", "df.count() == 0"),
        ("for-loop", "Pyspark-Coding", "Avoid inefficient loop-based transformations", "for row in df.collect()"),
        ("select", "Pyspark-Coding", "Select only necessary columns", "df.select('*')"),
        ("merge", "Pyspark-Coding", "Partition/filter data before merge", "MERGE INTO table"),
        ("rdd", "Pyspark-Coding", "Avoid RDD API in favor of DataFrame API", "sc.parallelize()"),
        ("cross-partition", "Pyspark-Coding", "Avoid cross-partition writes", "repartition()"),
        ("un-necessary-distinct", "Pyspark-Sql", "Avoid blind distinct / dropDuplicates", "distinct()"),
        ("multi-source-read", "Pyspark-Sql", "Avoid reading same source repeatedly", "spark.read"),
        ("hard-coded-filter", "Pyspark-Sql", "Avoid hardcoded site-specific filters", "WHERE site_id = 123"),
        ("shuffle-partition", "Pyspark-Config", "Use spark.sql.shuffle.partitions=auto", "shuffle.partitions"),
        ("join-groupby", "Pyspark-Sql", "Avoid unnecessary join-groupby patterns", "JOIN ... GROUP BY"),
        ("multi-group-by", "Pyspark-Sql", "Avoid unnecessary chained group/window patterns", "groupBy().groupBy()"),
        ("destructive-statements", "Pyspark-Sql", "Avoid destructive SQL without safe filtering", "DROP TABLE"),
        ("hard-coded-secrets", "Pyspark-Coding", "Avoid hardcoded secrets, credentials", "password = 'secret'"),
        ("modularized-code", "Pyspark-Coding", "Wrap repeated read/write logic in reusable functions", "def read_table"),
        ("incremental-data-pull", "Pyspark-Coding", "Filter on timestamp for incremental loads", "WHERE created_ts >"),
        ("documented-code", "Pyspark-Coding", "Document functions and notebook logic", "def my_func():"),
        ("manual-finetuning", "Pyspark-Optimization", "Avoid manual repartition/coalesce when unnecessary", "repartition(100)"),
        ("jdbc-sql-server", "Pyspark-Optimization", "Prefer SQL Server Spark connector over generic JDBC", "jdbc:sqlserver"),
        ("jdbc-optimize", "Pyspark-Optimization", "Parallelize JDBC reads", "numPartitions"),
        ("event-hub-read", "Pyspark-Optimization", "Prefer Kafka protocol over EventHub connector", "eventhubs"),
        ("overwrite-avoid", "Pyspark-Sql", "Avoid overwrite mode on large tables", "mode='overwrite'"),
        ("avoid-multi-scan", "Pyspark-Coding", "Avoid repeated scans of same path with different filters", "spark.read.parquet"),
        ("multi-union-cache", "Pyspark-Coding", "Avoid multi-union on same source", "union()"),
        ("avoid-explode", "Pyspark-Coding", "Avoid explode+filter when higher-order functions suffice", "explode()"),
        ("avoid-schema", "Pyspark-Coding", "Avoid mergeSchema and inferSchema", "mergeSchema=true"),
        ("broadcast-join", "Pyspark-Optimization", "Broadcast small dimensions", "broadcast(small_df)"),
        ("checkpoint-long-lineage", "Pyspark-Optimization", "Use checkpoint/persist for long lineages", "checkpoint()"),
        ("avoid-repeated-spark-config", "Pyspark-Config", "Avoid repeatedly setting Spark configs in loops", "spark.conf.set"),
        ("avoid-redundant-transforms", "Pyspark-Coding", "Encapsulate repeated transformations in utility functions", "def transform"),
    ]
    
    try:
        for rule_name, category, description, sample in default_rules:
            spark.sql(f"""
                INSERT INTO {table_name} (rule_name, rule_description, rule_category, rule_sample, is_enabled)
                SELECT '{rule_name}', '{description}', '{category}', '{sample}', true
                WHERE NOT EXISTS (
                    SELECT 1 FROM {table_name} WHERE rule_name = '{rule_name}'
                )
            """)
        
        count = spark.sql(f"SELECT COUNT(*) as cnt FROM {table_name}").collect()[0].cnt
        logger.info(f"Seeded {count} rules into {table_name}")
    except Exception as e:
        logger.error(f"Failed to insert rules: {e}")
        raise

## Add Custom Rule Helper

In [ ]:
def add_rule(spark, rule_name=None, description=None, category=None, sample=None, enabled=True):
    """
    Add a custom rule interactively (or with parameters).
    
    Args:
        spark: SparkSession
        rule_name: Rule name (prompted if None)
        description: Rule description (prompted if None)
        category: Rule category (prompted if None)
        sample: Rule sample code (prompted if None)
        enabled: Whether rule is enabled (default True)
    """
    table_name = DB_TABLES["genai_rule_patterns"]
    
    if not rule_name:
        rule_name = input("Enter rule name: ")
    if not description:
        description = input("Enter rule description: ")
    if not category:
        category = input("Enter rule category: ")
    if not sample:
        sample = input("Enter rule sample code: ")
    
    try:
        # Check for duplicates
        existing = spark.sql(f"SELECT COUNT(*) as cnt FROM {table_name} WHERE rule_name = '{rule_name}'").collect()[0].cnt
        if existing > 0:
            print(f"Rule '{rule_name}' already exists")
            return
        
        # Insert new rule
        spark.sql(f"""
            INSERT INTO {table_name} (rule_name, rule_description, rule_category, rule_sample, is_enabled)
            VALUES ('{rule_name}', '{description}', '{category}', '{sample}', {str(enabled).lower()})
        """)
        
        # Get new rule_id
        new_id = spark.sql(f"SELECT MAX(rule_id) as id FROM {table_name}").collect()[0].id
        print(f"✓ Rule added with ID: {new_id}")
    except Exception as e:
        logger.error(f"Failed to add rule: {e}")
        raise

## Execute Setup

In [ ]:
print("[DB_Setup] Initializing tables...")
setup_tables(spark)
print("[DB_Setup] Seeding default rules...")
insert_rules(spark)
print("[DB_Setup] ✓ Setup complete")